# 04e_dim_position_ceiling

**Purpose:** build `dim_position_ceiling` — the cross-position scarcity
multiplier that makes a QB, a TE and a linebacker priceable in one currency.

**The problem it solves.** `mouserat_trade-bud` used to value a player by a
percentile taken across a whole *format* pool (SF for offense, IDP for
defense) and then summed the two as if they were the same unit. They are not:
an 80th-percentile IDP and an 80th-percentile QB come from unrelated pools, so
every mixed offense/IDP package was mispriced. The fix is to percentile
*within* a position and then scale positions against each other by something
real. This table is that something.

**Definition.** Value over replacement, where replacement is **the best free
agent at that position** — the true opportunity cost, since that is what a
roster spot costs you nothing to fill with.

```
vor(pos)     = max_fpts(pos) - best_free_agent_fpts(pos)
ceiling(pos) = vor(pos) ** CEILING_EXPONENT, rescaled so the largest = 100
```

Scarcity and scoring are both already in the VOR term: a position with deep
free agency gets a small VOR *and* is compared on the same points scale as
everyone else. Notably LB out-scores TE on raw points yet prices below it,
because LB replacement is nearly as good as LB best — the offense/defense
meta-scarcity, derived rather than asserted.

**The one knob, and why it exists.** `CEILING_EXPONENT = 0.5` (sqrt), set
2026-07-31. At `1.0` (raw VOR) the ordering is right but the magnitude is not
tradeable: the best WR in the league priced at 57.9, below RB48 and QB25, and
the top 50 assets league-wide contained zero WR, TE or IDP. A concave
transform is monotone, so it changes no position's *rank* — it only narrows
the gaps (WR 57.9 → 76.1, DL 11.7 → 34.3). `vor` is stored raw, so the
exponent can be re-set and the table re-derived without re-pulling anything.

**Why per conference.** This is a 28-team, dual-conference league with one
player copy per conference, so the two conferences are genuinely separate
player pools. Replacement is therefore a **14-team** computation, not 28.
Computing it league-wide understates free-agent depth and produces the
familiar units artifact (an implausibly low QB replacement).

**Inputs**
- `fact_fantrax_adp` (season 2026, week `PRE`) — the full active-roster player
  universe with projected FPts. Must be the Players-grid partition from `04a`'s
  `main_snapshot`; the retired `week='DRAFT'` board carried a 3-5x truncated
  offense pool and would silently corrupt every number here.
- `fact_roster_placement` — who is rostered, and in which conference
  (`team_key` prefix `A*`/`B*`).
- `dim_position` — the transformer table, for `position_raw -> position_group`.

**Output:** `data/dim_position_ceiling.parquet`, grain
`(snapshot_date, conference, position_group)`. `conference='ALL'` is the row
the app reads; `'A'`/`'B'` are kept for diagnostics and to show the two pools
agree. Small reference table, same shape of thing as `dim_pick_value_curve`.

In [1]:
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA, REVIEW

import numpy as np
import pandas as pd

# The universe partition this table is defined against. Pinned rather than
# "latest": week='DRAFT' is the retired, offense-truncated board and must never
# be picked up by accident.
SOURCE_SEASON = 2026
SOURCE_WEEK = "PRE"

# The seven roster-able position groups. OL/K/DEF etc. exist in dim_position
# but are not rosterable here, so they get no ceiling.
POSITIONS = ["QB", "RB", "WR", "TE", "DL", "LB", "DB"]

# Concavity applied to VOR before rescaling (1.0 = raw VOR, 0.5 = sqrt).
# The one deliberate knob in this table, set 2026-07-31. Raw VOR is a fair
# measure of *points* over replacement but overstates *tradeable* value at the
# extremes: on raw VOR the best WR in the league (57.9) priced below RB48 and
# QB25, and no WR/TE/IDP appeared in the top 50 assets league-wide. A concave
# transform preserves the ordering exactly — it is monotone — and narrows the
# gaps (WR 57.9 -> 76.1, DL 11.7 -> 34.3). Set back to 1.0 to see raw VOR.
CEILING_EXPONENT = 0.5

# Truncation alarm, mirroring 04a's MIN_ACTIVE_BY_POS. If the universe is short
# again for any reason, fail loudly here rather than emitting plausible-looking
# ceilings computed from a partial pool.
MIN_POOL = {"QB": 100, "RB": 180, "WR": 330, "TE": 165,
            "DL": 380, "LB": 300, "DB": 500}

In [2]:
# ---- Load + normalize -------------------------------------------------------
adp = pd.read_parquet(DATA / "fact_fantrax_adp.parquet")
adp = adp[(adp["season"] == SOURCE_SEASON) & (adp["week"] == SOURCE_WEEK)].copy()
if adp.empty:
    raise RuntimeError(
        f"no fact_fantrax_adp rows for season={SOURCE_SEASON} week={SOURCE_WEEK} "
        "— run 04a_fantrax_weekly_scrape.py first")

roster = pd.read_parquet(DATA / "fact_roster_placement.parquet")
roster["conference"] = roster["team_key"].str[0]

pos_map = pd.read_parquet(DATA / "dim_position.parquet")
pos_map = dict(zip(pos_map["position_raw"], pos_map["position_group"]))

# Multi-eligibility: position_raw carries tokens like "DL,LB" / "WR,DB". Explode
# so a player counts at EVERY position they can fill — that is how they are
# actually available to a roster slot, and it is how the replacement a manager
# faces at that slot is really determined. Route through dim_position rather
# than mapping codes inline (transformer-table rule).
univ = adp.assign(position_group=adp["position_raw"].fillna("").str.split(",")).explode("position_group")
univ["position_group"] = univ["position_group"].str.strip().map(pos_map)
univ = univ[univ["position_group"].isin(POSITIONS) & univ["fpts"].notna()]

short = {p: n for p, n in MIN_POOL.items()
         if univ.loc[univ["position_group"] == p, "scorer_id"].nunique() < n}
if short:
    raise RuntimeError(f"player universe is truncated at {short} — re-run 04a")

print(f"[ok] universe {univ['scorer_id'].nunique()} players, "
      f"{len(univ)} position-eligibilities")
print(f"[ok] rosters {roster['scorer_id'].nunique()} players across "
      f"{roster['team_key'].nunique()} teams / "
      f"{roster['conference'].nunique()} conferences")

[ok] universe 2265 players, 2342 position-eligibilities
[ok] rosters 556 players across 28 teams / 2 conferences


In [3]:
# ---- Value over replacement, per conference ---------------------------------
def _conference_vor(univ: pd.DataFrame, roster: pd.DataFrame, conf: str) -> pd.DataFrame:
    """One row per position: pool depth, free-agent depth, best, replacement, VOR.

    Free agent = in the Fantrax active-roster universe but on no team IN THIS
    CONFERENCE. The other conference's rosters are irrelevant — its copy of a
    player is a different asset that this conference's managers cannot acquire.
    """
    owned = set(roster.loc[roster["conference"] == conf, "scorer_id"])
    out = []
    for pos, grp in univ.groupby("position_group", sort=False):
        fa = grp[~grp["scorer_id"].isin(owned)]
        if fa.empty:
            # Would mean a position is fully rostered league-wide, so "what it
            # costs to replace him" has no answer. Real once (TE, under the
            # truncated board); a genuine occurrence needs a design decision,
            # not a silent zero.
            raise RuntimeError(f"conference {conf}: no free agents at {pos}")
        best_fa = fa.loc[fa["fpts"].idxmax()]
        out.append({
            "conference": conf,
            "position_group": pos,
            "pool_size": grp["scorer_id"].nunique(),
            "n_rostered": grp["scorer_id"].isin(owned).sum(),
            "n_free_agents": fa["scorer_id"].nunique(),
            "max_fpts": float(grp["fpts"].max()),
            "replacement_fpts": float(best_fa["fpts"]),
            "best_free_agent": best_fa["player_name"],
        })
    return pd.DataFrame(out)


per_conf = pd.concat(
    [_conference_vor(univ, roster, c) for c in sorted(roster["conference"].unique())],
    ignore_index=True,
)
per_conf["vor"] = per_conf["max_fpts"] - per_conf["replacement_fpts"]

# The 'ALL' row averages VOR across conferences BEFORE rescaling, so a player's
# price never depends on which conference copy is being traded.
rollup = (per_conf.groupby("position_group", as_index=False)
          .agg(pool_size=("pool_size", "max"),
               n_rostered=("n_rostered", "mean"),
               n_free_agents=("n_free_agents", "mean"),
               max_fpts=("max_fpts", "max"),
               replacement_fpts=("replacement_fpts", "mean"),
               vor=("vor", "mean")))
rollup["conference"] = "ALL"
rollup["best_free_agent"] = None

ceiling = pd.concat([per_conf, rollup], ignore_index=True)
# Compress, then rescale within each conference scope so every scope's top
# position = 100. `vor` itself is stored raw, so the transform stays auditable
# and CEILING_EXPONENT can be re-set without re-running the pull.
compressed = np.power(ceiling["vor"], CEILING_EXPONENT)
ceiling["ceiling"] = compressed / compressed.groupby(ceiling["conference"]).transform("max") * 100

# datetime64 to match every other snapshot table in the model (etl.TODAY is
# an ISO string; dim_pick_value_curve and the dynasty fact both store dates).
ceiling["snapshot_date"] = pd.Timestamp(etl.TODAY)
ceiling["source_season"] = SOURCE_SEASON
ceiling["source_week"] = SOURCE_WEEK
ceiling = ceiling[["snapshot_date", "conference", "position_group",
                   "pool_size", "n_rostered", "n_free_agents",
                   "max_fpts", "replacement_fpts", "best_free_agent",
                   "vor", "ceiling", "source_season", "source_week"]]
ceiling = ceiling.sort_values(["conference", "ceiling"], ascending=[True, False])

print(ceiling.to_string(index=False))

snapshot_date conference position_group  pool_size  n_rostered  n_free_agents  max_fpts  replacement_fpts  best_free_agent     vor    ceiling  source_season source_week
   2026-07-31          A             QB        118        56.0           62.0    411.35            20.120  Gardner Minshew 391.230 100.000000           2026         PRE
   2026-07-31          A             RB        209        96.0          113.0    458.56            70.930       Ty Johnson 387.630  99.538849           2026         PRE
   2026-07-31          A             WR        390       122.0          268.0    334.88           105.160      Josh Palmer 229.720  76.627266           2026         PRE
   2026-07-31          A             TE        206        60.0          146.0    274.63           100.200     Tyler Higbee 174.430  66.772020           2026         PRE
   2026-07-31          A             LB        393        66.0          327.0    350.50           260.000  Robert Spillane  90.500  48.095918           202

In [4]:
# ---- Sanity checks ----------------------------------------------------------
# These assert the SHAPE of the answer, not specific numbers — the numbers are
# supposed to move as projections and rosters do.
allrow = ceiling[ceiling["conference"] == "ALL"].set_index("position_group")

offense, idp = ["QB", "RB", "WR", "TE"], ["DL", "LB", "DB"]
assert allrow.loc[offense, "ceiling"].min() > allrow.loc[idp, "ceiling"].max(), (
    "every offense position should out-price every IDP position: offense scores "
    "more AND has thinner free agency. If this trips, look at the universe "
    "partition before touching this table.")
assert (allrow["ceiling"] > 0).all(), "a non-positive ceiling means replacement >= best"
assert abs(allrow["ceiling"].max() - 100) < 1e-9, "top position must rescale to exactly 100"

# The two conferences are independent samples of the same league. They should
# agree closely; a wide gap means one conference's roster capture is stale.
spread = (per_conf.pivot(index="position_group", columns="conference", values="vor")
          .assign(pct_gap=lambda d: (d.max(axis=1) - d.min(axis=1)) / d.max(axis=1) * 100))
print(spread.round(1).to_string())
if spread["pct_gap"].max() > 35:
    print(f"[warn] conferences disagree by {spread['pct_gap'].max():.0f}% at "
          f"{spread['pct_gap'].idxmax()} — check fact_roster_placement freshness")

print("\n[ok] all checks passed")

conference          A      B  pct_gap
position_group                       
DB               64.7   64.7      0.0
DL               37.3   55.8     33.2
LB               90.5  107.2     15.6
QB              391.2  402.0      2.7
RB              387.6  399.2      2.9
TE              174.4  174.4      0.0
WR              229.7  229.7      0.0

[ok] all checks passed


In [5]:
# ---- Load -------------------------------------------------------------------
# Replace-by-snapshot_date: re-running on the same day is idempotent, and each
# run keeps history so a ceiling shift is auditable against the roster/projection
# state that produced it.
path = DATA / "dim_position_ceiling.parquet"
total = etl.load_replace_partition(ceiling, path, part_cols=("snapshot_date",))

final = pd.read_parquet(path)
n_snaps = final["snapshot_date"].nunique()
print(f"[ok] {len(ceiling)} rows -> {path} ({total} total, {n_snaps} snapshots)")

[ok] 21 rows -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data\dim_position_ceiling.parquet (21 total, 1 snapshots)
